In [ ]:
pip install cptec-subsaz

In [ ]:
pip install basemap

In [ ]:
import subsaz.CPTEC_SUB as SUB
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.basemap import Basemap

In [ ]:
subsaz = SUB.model()


In [ ]:
#Data Condição Inicial (IC)
date = '20240626'

#variavel
var = 'prec'

#produto semanal escolhido
# week - semanal
# fort - 14 dias
# 3wks - 21 dias
# mnth - mensal
product = 'week'

#qual campo deseja plotar?
#'anomalies' - anomalias do campo
#'prob_positve_anomaly' - probabilidade de anomalias positivas
#'prob_terciles' - probabilidades de tercis
#'totals' - total previsto para o período/produto escolhido
field = 'anomalies'

#passo depende do produto escolhido
# para produtos semanais, são produzidos 4 campos 01:04
# para produtos com 14 dias, 2 produtos (01 e 02)
# para 21 dias e mensal, apenas 01 disponível
step = '01'

#Requisição dos dados de acordo com as variáveis solicitadas anteriormente
f = subsaz.load(date=date, var=var, step=step, product=product ,field=field)
f

In [ ]:
# Supondo que 'f' seja um Dataset xarray e 'prec' seja a variável de interesse
fig, ax = plt.subplots(figsize=(10, 6))

# Configurar o mapa
mapa = Basemap(projection='cyl', resolution='i', ax=ax,
               llcrnrlat=-60, urcrnrlat=15, llcrnrlon=-100, urcrnrlon=-30)

# Desenhar linhas de continentes
mapa.drawcoastlines()

# Desenhar as fronteiras dos países (opcional)
mapa.drawcountries()

# Configurar a escala de cores
cmap = plt.get_cmap('RdBu')
levels = np.arange(-30, 35, 5)

# Extraindo os dados de 'f' e ajustando as dimensões para contourf
lon, lat = np.meshgrid(f.lon.values, f.lat.values)
prec = f.prec[0, :, :].values  # Seleciona a primeira camada se 'prec' tiver 3 dimensões

# Plotando com contourf para uma escala de cores discreta
mappable = mapa.contourf(lon, lat, prec, levels=levels, cmap=cmap, extend='both')

# Adicionar a barra de cores
cbar = plt.colorbar(mappable, ax=ax)
cbar.set_ticks(levels)
cbar.set_label('Valores')

plt.title('Mapa de Anomalia de Precipitação, semana: '+date)
plt.show()

In [ ]:
# Inicializa o construtor
subsaz = SUB.model()
# Filtrar area definida
subsaz.dict['area']['reduce'] = True
subsaz.dict['area']['minlat'] = -34.44
subsaz.dict['area']['maxlat'] = -21.43
subsaz.dict['area']['minlon'] = 301.14
subsaz.dict['area']['maxlon'] = 320.57

# Requisição dos dados
f = subsaz.load(date='20250101', var='prec_ca', product='mnth' ,field='anomalies')

# Definir tamanho da figura
fig = plt.figure(figsize=(10,8))

# Setar figura unica
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())

# Colocar  Linhas de Borda dos paises e linhas costeiras
ax.add_feature(cfeature.COASTLINE,color='grey')
ax.add_feature(cfeature.BORDERS,color='grey')

# Definir Regiao do Brasil
ax.set_extent([-90,-30,10,-41], ccrs.PlateCarree())

# Setar estados do Brasil
states = cfeature.NaturalEarthFeature(category='cultural',
                                         name='admin_1_states_provinces_lines',
                                         scale='50m', facecolor='none')

# Colocar Estados Brasil
ax.add_feature(states, edgecolor='gray')

# Plotar variavel
f.prec_ca.sel(time="2025-01-01").plot()
plt.show()